Reste a mmettre les news dans graph  
faire une page dédié   
+page dédié avec indicateurs dur info société  

In [18]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import io
import json
import streamlit as st
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# historique finance

In [19]:
def compute_slope_betwween_2_points(
    x1: float, y1: float, x2: float, y2: float
) -> float:
    #  coefficient directeur
    """
    Calculates the directing coefficient (slope) of a line passing through two points.

    Args:
        x1 (float): x coordinate of first point.
        y1 (float): y coordinate of first point.
        x2 (float): x coordinate of second point.
        y2 (float): Y coordinate of the second point.

    Returns:
        float: The directing coefficient of the line.
    """
    if x1 == x2:
        raise ValueError(
            "The points have the same x coordinate. The line is vertical and its slope is indefinite."
        )

    slope = (y2 - y1) / (x2 - x1)
    # print(f"Calcul ==> ({y2} - {y1}) / ({x2} - {x1})")
    return slope


In [ ]:
def get_trends_events(x: pd.Series, y: pd.Series, degre: int, show_curve: bool = False):
    coefficients = np.polyfit(x, y, degre)

    # Créer un modèle polynomial à partir des coefficients
    polynome = np.poly1d(coefficients)

    # Calculer les valeurs lissées
    y_lisse = polynome(x)

    first_day = x[0]
    last_day = x[-1]

    first_value = y_lisse[0]
    last_value = y_lisse[-1]
    print(f"x1={first_day}, y1={first_value}, x2={last_day}, y2={last_value}")

    slope = compute_slope_betwween_2_points(
        x1=first_day, y1=first_value, x2=last_day, y2=last_value
    )

    if show_curve:
        # Tracer les données originales et la courbe lissée
        plt.scatter(x, y, color="blue", label="Points observés")
        plt.plot(
            x, y_lisse, color="firebrick", label=f"Régression polynomiale de degré {degre}"
        )
        # plt.set_xticklabels(df["Date_str"], rotation=90)
        plt.legend()
    return {
        "first_day": first_day,
        "last_day": last_day,
        "first_value": first_value,
        "last_value": last_value,
        "slope": slope,
    }


# __________________________________________________________________________________
# def get_trends_events(df: pd.DataFrame, cols: list[str], degre: int, show_curve: bool = False):
#     x = df.index.values
#     for col in cols:
#         y = df[col]
#         coefficients = np.polyfit(x, y, degre)

#         # Créer un modèle polynomial à partir des coefficients
#         polynome = np.poly1d(coefficients)

#         # Calculer les valeurs lissées
#         y_lisse = polynome(x)

#         first_day = x[0]
#         last_day = x[-1]

#         first_value = y_lisse[0]
#         last_value = y_lisse[-1]
#         print(f"x1={first_day}, y1={first_value}, x2={last_day}, y2={last_value}")

#         slope = compute_slope_betwween_2_points(
#             x1=first_day, y1=first_value, x2=last_day, y2=last_value
#         )


#     if show_curve:
#         # Tracer les données originales et la courbe lissée
#         plt.scatter(x, y, color="blue", label="Points observés")
#         plt.plot(
#             x, y_lisse, color="red", label=f"Régression polynomiale de degré {degre}"
#         )
#         # plt.set_xticklabels(df["Date_str"], rotation=90)
#         plt.legend()
#     return {
#         "first_day": first_day,
#         "last_day": last_day,
#         "first_value": first_value,
#         "last_value": last_value,
#         "slope": slope
#     }


# import matplotlib.pyplot as plt
# import numpy as np
# import pandas as pd

In [60]:

def get_trends_events(
    df: pd.DataFrame,
    cols: list[str],
    degre: int,
    show_curve: bool = False,
    colors: dict[str, str] = None,
):
    x = df.index.values

    return_slop = {}

    if show_curve:
        _, ax = plt.subplots(figsize=(14, 7))

    for col in cols:
        y = df[col]
        coefficients = np.polyfit(x, y, degre)

        # Créer un modèle polynomial à partir des coefficients
        polynome = np.poly1d(coefficients)
        y_lisse = polynome(x)

        first_day = x[0]
        last_day = x[-1]
        first_value = y_lisse[0]
        last_value = y_lisse[-1]

        slope = compute_slope_betwween_2_points(
            x1=first_day, y1=first_value, x2=last_day, y2=last_value
        )

        if show_curve:
            color = colors.get(col, None) if colors else None
            ax.scatter(x, y, label=f"{col} - Observé", alpha=0.5, color=color)
            ax.plot(x, y_lisse, label=f"{col} - Régression degré {degre}", color=color)

        return_slop[col + "_slope"] = slope

    if show_curve:
        ax.legend(df["Date"])
        ax.set_xticklabels(df["Date"].apply(lambda x: str(x.date())), rotation=90)
        ax.set_ylim(first_value - 50, last_value + 50)
        ax.set_ylabel("Price")
        ax.set_xlabel("Date")
        ax.set_title("Courbes de régression polynomiale")
        plt.tight_layout()
        plt.show()

    return return_slop


In [61]:
get_trends_events_graph(
    df=df_5year,
    cols=["Open", "High", "Low", "Close"], 
    degre=3,
    show_curve=True, 
    colors={
        "Open":"cornflowerblue",
        "High":"forestgreen",
        "Low":"firebrick",
        "Close":"dimgrey"
    }
)

{'Open_slope': np.float64(0.0522857453338676),
 'High_slope': np.float64(0.05351640838641091),
 'Low_slope': np.float64(0.05053776386508113),
 'Close_slope': np.float64(0.05171578950138634)}

In [2]:
from API.get_data.api_yahoo import get_historical_data

In [ ]:
df = get_historical_data("MSFT", "5y")#business_ticker, "5y")
# df.to_csv("msft_14_07_temp.csv")

In [7]:
df_5year=pd.read_csv("msft_14_07_temp.csv").drop(["Unnamed: 0"], axis = 1)

In [7]:
df_5year = df.copy()

In [8]:
df

,Date,Year,Month,Day,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2020-07-13 00:00:00-04:00,2020,7,13,205.490228,206.754908,197.844707,198.390823,38135600,0.0,0.0
1,2020-07-14 00:00:00-04:00,2020,7,14,197.490191,200.096185,193.562034,199.617142,37591800,0.0,0.0
2,2020-07-15 00:00:00-04:00,2020,7,15,200.776454,202.472270,196.436327,199.320160,32179400,0.0,0.0
3,2020-07-16 00:00:00-04:00,2020,7,16,196.790781,197.078209,193.830300,195.372818,29940700,0.0,0.0
4,2020-07-17 00:00:00-04:00,2020,7,17,195.899756,196.445858,192.948851,194.376404,31635300,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
1251,2025-07-07 00:00:00-04:00,2025,7,7,497.380005,498.750000,495.230011,497.720001,13981600,0.0,0.0
1252,2025-07-08 00:00:00-04:00,2025,7,8,497.239990,498.200012,494.109985,496.619995,11846600,0.0,0.0
1253,2025-07-09 00:00:00-04:00,2025,7,9,500.299988,506.779999,499.739990,503.510010,18659500,0.0,0.0
1254,2025-07-10 00:00:00-04:00,2025,7,10,503.049988,504.440002,497.750000,501.480011,16492100,0.0,0.0


In [62]:
# get_trends_events(
#     df=df_5year,
#     cols=["Open", "High", "Low", "Close"], 
#     degre=3,
#     show_curve=True)

In [ ]:
import plotly.graph_objects as go
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

def compute_slope_betwween_2_points(
    x1: float, y1: float, x2: float, y2: float
) -> float:
    #  coefficient directeur
    """
    Calculates the directing coefficient (slope) of a line passing through two points.

    Args:
        x1 (float): x coordinate of first point.
        y1 (float): y coordinate of first point.
        x2 (float): x coordinate of second point.
        y2 (float): Y coordinate of the second point.

    Returns:
        float: The directing coefficient of the line.
    """
    if x1 == x2:
        raise ValueError(
            "The points have the same x coordinate. The line is vertical and its slope is indefinite."
        )

    slope = (y2 - y1) / (x2 - x1)
    # print(f"Calcul ==> ({y2} - {y1}) / ({x2} - {x1})")
    return slope


def get_trends_events_graph(
    df: pd.DataFrame,
    cols: list[str],
    degre: int,
    show_curve: bool = False,
    colors: dict[str, str] = None
):
    x = df.index.values
    return_slop = {}

    if show_curve:
        fig = go.Figure()

    for col in cols:
        y = df[col].values
        coefficients = np.polyfit(x, y, degre)
        polynome = np.poly1d(coefficients)
        y_lisse = polynome(x)

        first_day, last_day = x[0], x[-1]
        first_value, last_value = y_lisse[0], y_lisse[-1]

        slope = compute_slope_betwween_2_points(
            x1=first_day, y1=first_value, x2=last_day, y2=last_value
        )

        return_slop[col + "_slope"] = slope

        if show_curve:
            color = colors.get(col) if colors else None

            # Points observés
            fig.add_trace(go.Scatter(
                x=df["Date"],#.apply(lambda x : str(x.date())),
                y=y,
                mode='markers',
                name=f"{col} - Observé",
                marker=dict(color=color),
                opacity=0.5
            ))

            # Régression polynomiale
            fig.add_trace(go.Scatter(
                x=df["Date"],#.apply(lambda x : str(x.date())),
                y=y_lisse,
                mode='lines',
                name=f"{col} - polynomial modeling of {degre}",
                line=dict(color=color)
            ))

    if show_curve:
        fig.update_layout(
            title="Overview of Market Trends and Polynomial Modeling",
            xaxis_title="Date",
            yaxis_title="Price",
            height=600,
            width=1000,
            margin=dict(
                l=10,
                r=10,
                b=10,
                t=50,
                pad=4
            ),
        )
        fig.show()

    return return_slop



In [32]:
import json
import io
with open(r"C:\Users\cleme\Documents\Ynov\M2\Projet Master\Projet Bourse\NEW\Projet_master\data\output\all_data_regroup\anf_data_2025_07_21.json", "r", encoding="utf-8") as f:
    output_data = json.load(f)

In [34]:
df_5year = pd.read_json(io.StringIO(output_data["historical_stock_info"]["5y_historic"]))

In [42]:
get_trends_events_graph(
    df=df_5year,
    cols=["Open", "High", "Low", "Close"], 
    degre=3,
    show_curve=True, 
    colors={
        "Open":"cornflowerblue",
        "High":"forestgreen",
        "Low":"firebrick",
        "Close":"dimgrey"
    }
)

{'Open_slope': np.float64(0.0522857453338676),
 'High_slope': np.float64(0.05351640838641091),
 'Low_slope': np.float64(0.05053776386508113),
 'Close_slope': np.float64(0.05171578950138634)}

In [ ]:
def historic_stock_candle_stick_chart(df:pd.DataFrame, increasing_color:str="3D9970", decreasing_color:str="FF4136", height:int, width:int)
    fig = go.Figure(data=[go.Candlestick(x=df['Date'],
                open=df['Open'],
                high=df['High'],
                low=df['Low'],
                close=df['Close'])])

    fig.data[0].increasing.fillcolor = increasing_color
    fig.data[0].increasing.line.color = increasing_color
    fig.data[0].decreasing.fillcolor = decreasing_color
    fig.data[0].decreasing.line.color = decreasing_color

    fig.update_layout(
        # title="Historical Stock Performance",
        xaxis_title="Date",
        yaxis_title="Price",
        height=height,
        width=width,
        # margin=dict(
        #     l=25,
        #     r=10,
        #     b=50,
        #     t=50,
        #     pad=4
        # ),
        # paper_bgcolor="LightSteelBlue",
    )

    return fig

In [59]:
fig = go.Figure(data=[go.Candlestick(x=df_5year['Date'],
                open=df_5year['Open'],
                high=df_5year['High'],
                low=df_5year['Low'],
                close=df_5year['Close'])])

fig.data[0].increasing.fillcolor = '#3D9970'
fig.data[0].increasing.line.color = '#3D9970'
fig.data[0].decreasing.fillcolor = '#FF4136'
fig.data[0].decreasing.line.color = '#FF4136'

fig.update_layout(
    # title="Historical Stock Performance",
    xaxis_title="Date",
    yaxis_title="Price",
    height=600,
    width=1000,
    margin=dict(
        l=50,
        r=5,
        b=35,
        t=25,
        pad=4
    ),
    # paper_bgcolor="LightSteelBlue",
)

fig.show()

____

# info societe

In [4]:
# Read JSON file
import json
with open(r"C:\Users\cleme\Documents\Ynov\M2\Projet Master\Projet Bourse\NEW\Projet_master\data\output\all_data_regroup\anf_data_2025_07_09.json", "r", encoding="utf-8") as f:
    output_data = json.load(f)

# Use the output_data
pd.read_json(output_data["info_company"])


C:\Users\cleme\AppData\Local\Temp\ipykernel_31708\971176017.py:7: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  pd.read_json(output_data["info_company"])


,country,currency,finnhubIndustry,marketCapitalization,competitor_lst,begin_analyst_period,end_analyst_period,delta_number_days_analyst,mean_sell,mean_strongBuy,mean_strongSell,insider_sentiment
0,US,USD,Retail,4144.492192,"ROST, BURL, GAP, URBN, BOOT, ANF, BKE, FL, AEO...",2025-04-01,2025-07-01,91,0,1,0,1


# indicateur up down

In [5]:
idincateur = pd.read_json(output_data["historical_stock_info"]["today_analyse_price"])
df = idincateur.reset_index(drop=True)

C:\Users\cleme\AppData\Local\Temp\ipykernel_31708\2866061094.py:1: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  idincateur = pd.read_json(output_data["historical_stock_info"]["today_analyse_price"])


In [17]:
df.columns

Index(['Date', 'Close', 'Volume', 'this_month_most_direction',
       'last_month_most_direction', 'year_to_date_direction',
       'one_year_rolling_period_most_direction',
       'five_year_rolling_period_most_direction', 'this_month_trend',
       'last_month_trend', 'year_to_date_trend', 'one_year_rolling_trend',
       'five_year_rollingtrend', 'this_month_volume', 'last_month_volume',
       'year_to_date_volume', 'one_year_rolling_volume',
       'five_year_rolling_volume', 'analyst_recommendation_of_the_month'],
      dtype='object')

In [ ]:
# # Liste des colonnes de tendances de prix
# price_columns = [
#      #"this_month_most_direction",  "last_month_most_direction", "year_to_date_direction",
# #     "one_year_rolling_period_most_direction", "five_year_rolling_period_most_direction",
#      "this_month_trend", #"last_month_trend", "year_to_date_trend",
# #     "one_year_rolling_trend", "five_year_rollingtrend"
# ]

# # Liste des colonnes de tendances de volume
# volume_columns = [
#     "this_month_volume", "last_month_volume", "year_to_date_volume",
#     "one_year_rolling_volume", "five_year_rolling_volume"
# ]

In [13]:
def make_indicator(
        label:str,
        trend_text:str,
        color:str,
        row_number:int,
        col_number:int,
        background_color:str="white"
    )->go.Indicator:
    """
    Create an indicator graph that displays only a colored text string, without showing any numeric value.
    The numerical value is hidden by setting its font color to white (matching the background),
    while the trend text is styled using the specified color

    Input:
        - label: The label describing the indicator (displayed below the trend text).
        - trend_text: The trend text to display (e.g., "Bullish", "Down", etc.).
        - color: The color to apply to the trend text.
        - row_number: number of the row where the graph will be display
        - col_number: number of the col where the graph will be display
        - background_color : the color for the number value 

    Output:
        - go.Indicator: A Plotly Indicator object with formatted text and hidden numeric value.
    """
    return go.Indicator(
        mode="number",  # requis pour afficher le bloc
        value=0,  # valeur neutre
        number={'font': {'color': background_color}},  # rend la valeur invisible
        title={
            "text": f"<span style='font-size:13px'>{label}</span><br><span style='color:{color}; font-size:20px; font-weight:bold'>{trend_text}</span>"
        },
        domain = {'row':row_number, 'column':col_number}
        # domain={'x': [0, 1], 'y': [0, 1]}
    )


In [ ]:
def price_and_volume_kpi(
    col_price: str,
    col_volume: str,
    background_color: str
) -> go.Figure:
    """
    Display two KPI indicators using Plotly: one for the price trend and one for the volume trend.
    The indicators are styled with color-coded text and arrow to visually represent the current state,
    and the numerical value is hidden by matching the background color.

    Inputs:
        - col_price (str): Column name in the DataFrame representing the price trend.
        - col_volume (str): Column name in the DataFrame representing the volume trend.
        - background_color (str): Background color used to hide the number value.

    Output:
        - go.Figure: A Plotly figure containing the two styled indicators.
    """
    # Mapping de couleur
    color_map = {
        "Up": "forestgreen", "Down": "firebrick", "Bearish": "firebrick", "Bullish": "forestgreen",
        "Stagnant": "dimgrey", "Very bearish": "darkred", "Very bullish": "darkgreen"
    }
    icon_map = {
        "Up": "⇧", "Down": "⇩", "Bearish": "↘", "Bullish": "↗",
        "Stagnant": "→", "Very bearish": "⇩", "Very bullish": "⇧"
    }
    fig = go.Figure()

    #price
    price_value = df.loc[0, col_price]
    #volume
    volume_value = df.loc[0, col_volume]

    fig.add_trace(
        make_indicator(
            label= "Price trend",#col_price.replace("_", " ").title(), 
            trend_text= price_value + " " + icon_map.get(price_value, "black"), 
            color= color_map.get(price_value, "black"),
            row_number= 0,
            col_number=0,
            background_color = background_color
        )
    )

    fig.add_trace(
        make_indicator(
            label= "Volume",#col_volume.replace("_", " ").title(), 
            trend_text= volume_value + " " + icon_map.get(volume_value, "black"), 
            color= color_map.get(volume_value, "black"),
            row_number= 0,
            col_number=1,
            background_color = background_color
        )
    )

    fig.update_layout(
        width=450,
        height=100,
        margin=dict(
            l=25,
            r=25,
            b=25,
        #     t=100,
        #     pad=4
        ),
        paper_bgcolor = background_color,
        grid = {'rows': 1, 'columns': 2, 'pattern': "independent"},
    )
    # fig.show()
    return fig

In [233]:
# for the day
price_and_volume_kpi(
    col_price = "this_month_trend",
    col_volume = "this_month_volume",
    background_color = "lightgray"
)

In [10]:
price_and_volume_kpi(
    col_price = "this_month_trend",
    col_volume = "this_month_volume",
    background_color = "lightgray"
)

In [32]:
# fig = go.Figure()

# # fig.add_trace(go.Indicator(
# #     mode = "number+delta",
# #     value = 200,
# #     domain = {'x': [0, 0.5], 'y': [0, 0.5]},
# #     delta = {'reference': 400, 'relative': True, 'position' : "top"}))

# fig.add_trace(go.Indicator(
#     mode = "number+delta",
#     value = 350,
#     delta = {'reference': 400, 'relative': True},
#     domain = {'row':0, 'column':0}))

# fig.add_trace(go.Indicator(
#     mode = "number+delta",
#     value = 450,
#     title = {"text": "Accounts<br><span style='font-size:0.8em;color:gray'>Subtitle</span><br><span style='font-size:0.8em;color:gray'>Subsubtitle</span>"},
#     delta = {'reference': 400, 'relative': True},
#     domain = {'row':0, 'column':1}))

# fig.update_layout(
#     grid = {'rows': 1, 'columns': 2, 'pattern': "independent"},
#     # template = {'data' : {'indicator': [{
#     #     'title': {'text': "Speed"},
#     #     'mode' : "number+delta+gauge",
#     #     'delta' : {'reference': 90}}]
#     #                      }})
# )
# fig.show()

___

# Country news

In [ ]:
# pour gerer erreur lors lecteure json
# cn = pd.read_json(io.StringIO(output_data["news"]["country_news"])) 

# regler probleme des vues datafrmes avec les .loc ==> faire les modifs avant de gérer les vues

In [ ]:
cn = pd.read_json(io.StringIO(output_data["news"]["country_news"]))

In [156]:
cn["publishedAt"] = pd.to_numeric(cn["publishedAt"], errors='coerce')

# Unix timestamp in milliseconds 'unit='ms''
cn["publishedAt"] = pd.to_datetime(cn["publishedAt"], unit='ms')

cn["sentiment"] = cn["sentiment"].apply(lambda x : "negatif" if x ==0 else "positif")
uyuy = cn[["publishedAt", "headline", "summary", "source", "url", "sentiment"]]

# COMPANY news

In [147]:
a = pd.read_json(io.StringIO(output_data["news"]["company_news"]))

a["publishedAt"] = pd.to_numeric(a["publishedAt"], errors='coerce')

# Unix timestamp in milliseconds 'unit='ms''
a["publishedAt"] = pd.to_datetime(a["publishedAt"], unit='ms')

In [148]:
b = a[["publishedAt", "headline", "summary", "source", "url", "sentiment"]]
b["sentiment"] = b["sentiment"].apply(lambda x : "negatif" if x ==0 else "positif")

C:\Users\cleme\AppData\Local\Temp\ipykernel_19504\126627495.py:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [ ]:
# st.write(b)

In [11]:
# company_news = b

# fig = go.Figure(data=[go.Table(
#     header=dict(values=list(company_news.columns),
#                 fill_color='paleturquoise',
#                 align='left'),
#     cells=dict(values=company_news.transpose().values.tolist(),
#                fill_color='lavender',
#                align='left'))
# ])

# fig.update_layout(
#     title="News about the company with an sentiment analysis",
#     height=600,
#     width=1000,
#     margin=dict(
#         l=10,
#         r=10,
#         b=50,
#         t=50,
#         pad=4
#     ),
#     # paper_bgcolor="LightSteelBlue",
# )

# fig.show()

___

# Options

In [7]:
data = output_data["options"]

In [8]:
from typing import List

In [9]:
def bar_chart_options(   
        graph_title: str,
        left_bar_y_buy: List[float], left_bar_y_sell: List[float],
        left_title: str, left_yaxis_title: str,
        right_bar_y_buy: List[float], right_bar_y_sell: List[float],
        right_title: str, right_yaxis_title: str
    ) -> go.Figure:
    """
        Creates a dual bar chart with two subplots showing buy/sell volumes and counts.

    Args:
        graph_title (str): Title of the entire graph.
        left_bar_y_buy (List[float]): Y-values for 'Buy' bar in the left subplot.
        left_bar_y_sell (List[float]): Y-values for 'Sell' bar in the left subplot.
        left_title (str): Title of the left subplot.
        left_yaxis_title (str): Y-axis label for the left subplot.
        right_bar_y_buy (List[float]): Y-values for 'Buy' bar in the right subplot.
        right_bar_y_sell (List[float]): Y-values for 'Sell' bar in the right subplot.
        right_title (str): Title of the right subplot.
        right_yaxis_title (str): Y-axis label for the right subplot.

    Returns:
        Figure: Plotly Figure with two bar charts.
"""

    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=(left_title, right_title),
                        horizontal_spacing=0.15)

    # Left chart ==> Volume option
    fig.add_trace(
        go.Bar(name="Buy", x=["Buy"], y=left_bar_y_buy, marker_color="forestgreen", showlegend=False),
        row=1, col=1
    )
    fig.add_trace(
        go.Bar(name="Sell", x=["Sell"], y=left_bar_y_sell, marker_color="firebrick", showlegend=False),
        row=1, col=1
    )
    
    # Right chart ==> Number of options
    fig.add_trace(
        go.Bar(name="Buy", x=["Buy"], y=right_bar_y_buy, marker_color="forestgreen", showlegend=False),
        row=1, col=2
    )
    fig.add_trace(
        go.Bar(name="Sell", x=["Sell"], y=right_bar_y_sell, marker_color="firebrick", showlegend=False),
        row=1, col=2
    )
    go.Bar(name="Buy", x=["Buy"], y=data["nb_option_buy"]),
    go.Bar(name="Sell", x=["Sell"], y=data["nb_option_sell"])


    fig.update_layout(
        title_text=graph_title, 
        width=700,
        height=300,
        margin=dict(l=40, r=40, b=40, t=60),
        showlegend=False,
        # width=500,
        # height=250,
        # margin=dict(
        #     l=40,
        #     r=25,
        #     b=25,
        # #     t=100,
        # #     pad=4
        # ),
        paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', # No background color 
        yaxis=dict(title=left_yaxis_title),
        yaxis2=dict(title=right_yaxis_title, anchor="x2")
    )
    return fig

In [10]:
bar_chart_options(
    graph_title= "Buy/Sell options Comparison",
    # left
    left_bar_y_buy=data["buy_mean_volume"], left_bar_y_sell=data["sell_mean_volume"],
    left_title="Mean options volume",
    left_yaxis_title="Volume",

        
    right_bar_y_buy= data["nb_option_buy"], right_bar_y_sell=data["nb_option_sell"],
    right_title="Number of different options",
    right_yaxis_title="Number of options"
)


In [11]:
def graph_comparate_actual_price_with_option_price(
    cluster_buy: str,
    cluster_sell: str,
    background_color: str
) -> go.Figure:
    """
    Generates a dual-indicator Plotly chart comparing buy and sell option price clusters.
    Each cluster is visually styled based on its valuation category using a predefined color mapping.

    Args:
        cluster_buy: Valuation label for the buy option price  
        cluster_sell: Valuation label for the sell option price 
        background_color: Background color of the chart 
        
    Returns:
        go.Figure: A Plotly figure object containing two side-by-side indicators.
    """
    # Mapping de couleur
    color_map = {
        "Very undervalued": "darkgreen", 
        "Slightly undervalued": "forestgreen", # green
        "Price within spread": "dimgrey", 
        "Slightly overvalued": "firebrick", # red
        "Very overvalued": "darkred", 
    }

    fig = go.Figure()
    fig.add_trace(
        make_indicator(
            label= "Buy Price Option",#col_price.replace("_", " ").title(), 
            trend_text= cluster_buy ,#+ " " + icon_map.get(cluster_buy, "black"), 
            color= color_map.get(cluster_buy, "black"),
            row_number= 0,
            col_number=0,
            background_color = background_color
        )
    )

    fig.add_trace(
        make_indicator(
            label= "Sell Price Option",#col_volume.replace("_", " ").title(), 
            trend_text= cluster_sell ,#+ " " + icon_map.get(cluster_sell, "black"), 
            color= color_map.get(cluster_sell, "black"),
            row_number= 0,
            col_number=1,
            background_color = background_color
        )
    )

    fig.update_layout(
        width=500,
        height=100,
        margin=dict(
            l=25,
            r=25,
            b=25,
        #     t=100,
        #     pad=4
        ),
        paper_bgcolor = background_color,
        grid = {'rows': 1, 'columns': 2, 'pattern': "independent"},
    )
    # fig.show()
    return fig

In [14]:
graph_comparate_actual_price_with_option_price(
    cluster_buy = output_data["options"]["cluster_buy"][0],
    cluster_sell = output_data["options"]["cluster_sell"][0],
    background_color = "lightgray"
)

# Recommendation analyst

In [211]:
from typing import Dict, Any

In [230]:
def analyst_price_recommendation(options_data:Dict[str, Any])->go.Figure:
    fig = go.Figure(go.Indicator(
        mode="gauge+number",
        value=options_data["current_price"],
        title={'text': "Prix actuel vs Cibles analystes"},
        gauge={
            'axis': {'range': [options_data["low_analyst_price_targets"], options_data["high_analyst_price_targets"]]},
            'bar': {'color': "black"},
            'steps': [
                {'range': [options_data["low_analyst_price_targets"], options_data["median_analyst_price_targets"]], 'color': "darkorange"},
                {'range': [options_data["median_analyst_price_targets"], options_data["mean_analyst_price_targets"]], 'color': "dimgrey"},
                {'range': [options_data["mean_analyst_price_targets"], options_data["high_analyst_price_targets"]], 'color': "forestgreen"}
            ],
            'threshold': {
                'line': {'color': "red", 'width': 4},
                'thickness': 0.75,
                'value': options_data["current_price"]
            }
        }
    ))

    # Add an legend for each colors
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(size=12, color='darkorange'), name="LOWEST price to MEDIAN price"
    ))
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(size=12, color='dimgrey'), name="MEDIAN price to MEAN price"
    ))
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(size=12, color='forestgreen'), name="MEAN price to HIGHEST price"
    ))
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(size=12, color='red'), name="Actual price"
    ))

    fig.update_layout(
        width=450,
        height=300,
        # No background color
        paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.45,
            xanchor="center",
            x=0.5
        ),
        margin=dict(t=40, b=10)
    )

    return fig


In [231]:
analyst_price_recommendation(output_data["options"])

In [ ]:
sentiment_analyst = output_data["historical_stock_info"]["breakdown_of_analyst_recommendation"]["distribution_of_recommendations"]
# positive_sentiment = round(float(sentiment_analyst["positif"]), 1)
# negative_sentiment = round(float(sentiment_analyst["negatifs"]), 1)

In [248]:
def breakdown_of_sentiment_analyst(sentiment_analyst:dict[str, float])->go.Figure:
    """
    Generates a donut chart showing the distribution of positive and negative analyst sentiments.

    Args:
        - sentiment_analyst : Dictionary containing sentiment values with keys 'positif' and 'negatifs' (values between 0 and 1 or as percentages).

    Returns:
        fig: A Plotly donut chart visualizing positive vs. negative analyst sentiment.
    """
    positive_sentiment = round(float(sentiment_analyst["positif"]), 1)
    negative_sentiment = round(float(sentiment_analyst["negatifs"]), 1)
    
    labels = ['Positive', 'Negative']
    values = [positive_sentiment, negative_sentiment]
    colors = ['forestgreen', 'firebrick']

    fig = go.Figure(
        data=[go.Pie(
            labels=labels,
            values=values,
            marker=dict(colors=colors),
            textinfo='percent',
            hole=0.65,  # 0 = full pie, 0.5 = donut
            hoverinfo='label+percent'
        )]
    )

    fig.update_layout(
        title="Analyst sentiment distribution",
        paper_bgcolor='rgba(0,0,0,0)',
        showlegend=True,
        height=300,
        width=300
    )

    return fig


In [249]:
breakdown_of_sentiment_analyst(output_data["historical_stock_info"]["breakdown_of_analyst_recommendation"]["distribution_of_recommendations"])

In [ ]:
a = pd.read_json(io.StringIO(output_data["news"]["company_news"]))